# MLflow Model Registry — Self-Teaching Notebook

Practical, current MLflow 3.x workflow for promoting models from experiments to a versioned registry.

### How to use this notebook
- 📘 **Teaching cells** explain the concept.
- ✍️ **YOUR TURN cells** have blanks (`____`) for you to fill in *before* running.
- ✅ **Solution** cells are hidden right below — only peek after you try!

### Mental model
- A **run** produces a trained model (in an experiment).
- **Registering** copies that model into a named, **versioned** registry entry.
- **Aliases** (like `champion`, `production`) are movable pointers to a specific version — this is how you promote/deploy.

## 1. Connect to the tracking + registry server

The `MlflowClient` is our programmatic handle to both the **Tracking** store (experiments & runs) and the **Registry** (registered models & versions). Both live in our local `sqlite:///mlflow.db`.

⚠️ **Run this cell first every time** — after a kernel/Codespace restart, `mlflow`, `client`, etc. must be re-imported and reconnected.

In [1]:
import mlflow
from mlflow import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
print("mlflow version:", mlflow.__version__)

2026/08/13 17:29:42 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/13 17:29:42 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


mlflow version: 3.1.4


### ✍️ YOUR TURN #1 — list the experiments

Fill in the client method that lists experiments (hint: starts with `search_`).

In [2]:
# TODO: replace ____ with the correct method
experiments = client.search_experiments()
for e in experiments:
    print(e.experiment_id, e.name)

1 nyc-taxi-experiment
0 Default


<details><summary>✅ Solution</summary>

```python
experiments = client.search_experiments()
for e in experiments:
    print(e.experiment_id, e.name)
```
</details>

## 2. Find the best runs in the experiment

Search runs in `nyc-taxi-experiment`, filter by `rmse`, and order ascending so the best (lowest RMSE) is first.

In [3]:
from mlflow.entities import ViewType

experiment = client.get_experiment_by_name("nyc-taxi-experiment")

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="metrics.rmse < 7",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"],
)

for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 6b4bde3c50a04856ac52881fdf79608f, rmse: 6.5221
run id: 03cbeaaa7f0a4174ab7dd125b72bfd0e, rmse: 6.5261
run id: 7e20c5ee67f543c0aeb04dc6fe5e5644, rmse: 6.6725


## 3. Register a model version

**Registering** puts a run's trained model under a named, versioned registry entry. Registering again under the same name creates version 2, 3, ...

We use `runs:/{run_id}/model` as the source. This works as long as the run has a linked model (all our autolog runs do).

> If you ever see `Unable to find a logged_model with artifact_path model under run ...`, that run has no linked model — use a run that does, or register from a logged-model URI `models:/{model_id}`.

In [4]:
model_name = "nyc-taxi-regressor"

# Best run from section 2
best_run_id = runs[0].info.run_id
model_uri = f"runs:/{best_run_id}/model"

result = mlflow.register_model(model_uri=model_uri, name=model_name)
print("Registered:", result.name, "version", result.version)

2026/08/13 17:30:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/13 17:30:02 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
Successfully registered model 'nyc-taxi-regressor'.
2026/08/13 17:30:02 WARNING mlflow.tracking._model_registry.fluent: Run with id 6b4bde3c50a04856ac52881fdf79608f has no artifacts at artifact path 'model', registering model based on models:/m-852451047699472da7f548bc4b5ce349 instead


Registered: nyc-taxi-regressor version 1


Created version '1' of model 'nyc-taxi-regressor'.


### ✍️ YOUR TURN #2 — register a *second* version

Register the **second-best** run (`runs[1]`) under the **same** `model_name` so we get a version 2 to compare later.

In [5]:
second_run_id = runs[2].info.run_id
second_uri = f"runs:/{second_run_id}/model"
result2 = mlflow.register_model(model_uri=second_uri, name=model_name)
print("Registered:", result2.name, "version", result2.version)

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/08/13 17:30:08 WARNING mlflow.tracking._model_registry.fluent: Run with id 7e20c5ee67f543c0aeb04dc6fe5e5644 has no artifacts at artifact path 'model', registering model based on models:/m-04f4f734f69d4249be4ee4eb8c3d2383 instead


Registered: nyc-taxi-regressor version 2


Created version '2' of model 'nyc-taxi-regressor'.


<details><summary>✅ Solution</summary>

```python
second_run_id = runs[1].info.run_id
second_uri = f"runs:/{second_run_id}/model"
result2 = mlflow.register_model(model_uri=second_uri, name=model_name)
print("Registered:", result2.name, "version", result2.version)
```
</details>

## 4. List the versions of a registered model

Use `client.search_model_versions(f"name='{model_name}'")` to see all versions.

In [6]:
for v in client.search_model_versions(f"name='{model_name}'"):
    print(f"version: {v.version}, run: {v.run_id[:8]}")

version: 2, run: 7e20c5ee
version: 1, run: 6b4bde3c


## 5. Promote versions with **aliases**

An **alias** is a movable, human-friendly pointer to a version (think git tag pointing at a commit). We'll set `champion` → v1 and `challenger` → v2.

🔑 **Gotcha:** aliases do NOT show up on each version from `search_model_versions()`. They live on the **registered model** object as an `{alias: version}` dict — read them via `client.get_registered_model(name).aliases`.

In [7]:
client.set_registered_model_alias(name=model_name, alias="champion", version=1)
client.set_registered_model_alias(name=model_name, alias="challenger", version=2)

# Read aliases the reliable way
print("aliases:", client.get_registered_model(model_name).aliases)
for alias in ["champion", "challenger"]:
    mv = client.get_model_version_by_alias(model_name, alias)
    print(f"  {alias} -> version {mv.version}")

aliases: {'challenger': 2, 'champion': 1}
  champion -> version 1
  challenger -> version 2


## 6. Add metadata: tags and description

Attach **tags** (key/value) and a free-text **description** to a version — e.g. `validation_status`, who approved it, date.

🔑 **Gotcha:** there is no `get_model_version_tags()`. You **set** with dedicated methods but **read** tags/description off the version object: `client.get_model_version(name, version).tags` / `.description`.

In [8]:
from datetime import datetime

client.set_model_version_tag(name=model_name, version=1, key="validation_status", value="pending")
client.update_model_version(
    name=model_name,
    version=1,
    description=f"Champion candidate, registered on {datetime.today().date()}.",
)

mv = client.get_model_version(name=model_name, version=1)
print("tags:", mv.tags)
print("description:", mv.description)

tags: {'validation_status': 'pending'}
description: Champion candidate, registered on 2026-08-13.


## 7. Compare versions on an unseen test set

Real-world scenario: score both candidates (`champion`, `challenger`) on **fresh data they never saw** (March 2021) and let the numbers decide.

Steps:
1. Load the March test data.
2. Load the `DictVectorizer` preprocessor saved with the training run.
3. Preprocess the test set with it.
4. Load each model **by alias** and compare RMSE.

Key practical point: the preprocessor must match exactly how features were built at training time (here: separate `PULocationID` / `DOLocationID`). A mismatch here is one of the most common silent production bugs.

In [9]:
import pandas as pd
from sklearn.metrics import root_mean_squared_error


def read_dataframe(filename):
    df = pd.read_parquet(filename)
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df


def preprocess(df, dv):
    # IMPORTANT: match this project's features (separate location IDs, not PU_DO)
    categorical = ['PULocationID', 'DOLocationID']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(dicts)


test_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-03.parquet'
df_test = read_dataframe(test_url)
print("test rows:", len(df_test))

test rows: 80372


In [10]:
import pickle

# Download the preprocessor artifact logged with the best run
local_path = mlflow.artifacts.download_artifacts(
    run_id=best_run_id, artifact_path="preprocessor", dst_path="."
)
print("downloaded to:", local_path)

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

X_test = preprocess(df_test, dv)
y_test = df_test["duration"].values
print("X_test shape:", X_test.shape)

downloaded to: /workspaces/mlops-zoomcamp/02-experiment-tracking/preprocessor
X_test shape: (80372, 507)


### ✍️ YOUR TURN #3 — load a model **by alias** and score it

In production you rarely hardcode a version number — you load whatever the alias currently points to. The URI format is `models:/{name}@{alias}`. Complete the function.

In [12]:
def test_model(name, alias, X_test, y_test):
    # TODO: build the alias URI and load the model
    model = mlflow.pyfunc.load_model(f"models:/{model_name}@{alias}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

print(test_model(model_name, "champion", X_test, y_test))
print(test_model(model_name, "challenger", X_test, y_test))

{'rmse': 6.436859724151475}
{'rmse': 6.593170206471867}


<details><summary>✅ Solution</summary>

```python
def test_model(name, alias, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}@{alias}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

print(test_model(model_name, "champion", X_test, y_test))
print(test_model(model_name, "challenger", X_test, y_test))
```
</details>

## 8. Promote the winner to production

**Promotion = moving an alias.** Point a `production` alias at whichever version won on the test set. Aliases are mutable pointers, so if `production` already pointed elsewhere, this single call moves it.

> Practical reality: the registry does **not** deploy anything. It just labels versions. Your deployment/CI-CD reads these aliases (e.g. load `models:/nyc-taxi-regressor@production`) to do the real serving.

### ✍️ YOUR TURN #4 — set the `production` alias to the better version

Decide which version won in section 7, then point `production` at it.

In [19]:
winning_version = client.get_model_version_by_alias(model_name, 'champion').version   # the version with the lower test RMSE from section 7
client.set_registered_model_alias(name=model_name, alias="production", version=winning_version)

# Read aliases the reliable way
print("aliases now:", client.get_registered_model(model_name).aliases)

aliases now: {'challenger': 2, 'champion': 1, 'production': 1}


<details><summary>✅ Solution (example)</summary>

```python
winning_version = 1  # whichever had the lower test RMSE
client.set_registered_model_alias(name=model_name, alias="production", version=winning_version)
```
</details>

## 🎓 Recap — the practical workflow

1. **Register** a run's model: `mlflow.register_model(f"runs:/{run_id}/model", name)`.
2. List versions: `client.search_model_versions("name='...'")`.
3. **Aliases** are movable pointers: `client.set_registered_model_alias(name, alias, version)`.
4. **Read** aliases: `client.get_registered_model(name).aliases`.
5. **Set** tags/description with dedicated methods; **read** them off `client.get_model_version(name, version)`.
6. Load a model by alias: `mlflow.pyfunc.load_model(f"models:/{name}@{alias}")`.
7. **Promotion = moving an alias.** CI/CD (not the registry) does the real deploy.

### Cleanup helpers (handy when experimenting)
- Delete one version: `client.delete_model_version(name, version)`
- Delete a whole registered model: `client.delete_registered_model(name)`
- Remove an alias pointer only: `client.delete_registered_model_alias(name, alias)`

### Optional challenge
Delete an alias, confirm it's gone via `client.get_registered_model(name).aliases`, then try
`mlflow.pyfunc.load_model(f"models:/{name}@{deleted_alias}")`. What error do you get, and why is that the *correct* behaviour for a deployment system?